In [17]:
import math
from itertools import combinations

def P(p):
    """
    Baseline probability:
    (1 - (255/256)^p)^8
    """
    alpha = (255/256)**p
    return (1 - alpha)**8

In [18]:
def get_clusters(L):
    """
    Given a sorted list of indices L (subset of {1..7}), 
    returns a list of cluster sizes.
    Example: L=[1, 2, 4, 5, 6] -> [2, 3] (Cluster {1,2} size 2, Cluster {4,5,6} size 3)
    """
    if not L:
        return []
    
    clusters = []
    current_size = 1
    
    for i in range(1, len(L)):
        if L[i] == L[i-1] + 1:
            current_size += 1
        else:
            clusters.append(current_size)
            current_size = 1
    clusters.append(current_size)
    return clusters

def calculate_u_L(L, s):
    """
    Calculates u(L) = P(Single pattern matches AT LEAST ONE position in L).
    We use nested inclusion-exclusion on the subset K of L.
    
    P(X in Union S_j for j in L) = Sum_{K subset L} (-1)^(|K|+1) * (1/2)^(|Union R_k|)
    """
    n = len(L)
    u_val = 0.0
    
    # Iterate over all non-empty subsets K of L
    for r in range(1, n + 1):
        for K in combinations(L, r):
            # Calculate the exponent: Size of the union of intervals for K
            # Since s <= 16, non-adjacent clusters are disjoint.
            # Adjacent indices in K overlap.
            
            # 1. Get clusters of K to handle adjacency (e.g. {1,2} is 1 cluster)
            k_clusters = get_clusters(K)
            
            # 2. Calculate total bits: Sum(8*size + s - 8) for each cluster
            # Derivation: A cluster of size c has length 8(c-1) + s = 8c + s - 8
            total_bits = 0
            for size in k_clusters:
                total_bits += (8 * size + (s - 8))
            
            # 3. Add to probability sum with alternating sign
            term = math.pow(0.5, total_bits)
            
            if r % 2 == 1: # |K| is odd -> Add
                u_val += term
            else:          # |K| is even -> Subtract
                u_val -= term
                
    return u_val

def P_super(p, s):
    """
    Computes the upper bound for P(B).
    p: number of patterns
    s: size of super-character (bits)
    """
    # The set of bucket indices is {1, ..., 7}
    indices = range(1, 8)
    total_prob = 1.0
    
    # Iterate over all non-empty subsets L of {1..7}
    for r in range(1, 8):
        for L in combinations(indices, r):
            # L is a tuple, e.g., (1, 3, 4)
            
            # 1. Calculate u(L) for this specific geometric configuration
            u_L = calculate_u_L(L, s)
            
            # 2. Calculate the term (1 - u(L))^p
            # This is the prob that ALL p patterns fail to match L
            term_val = math.pow(1.0 - u_L, p)
            
            # 3. Add to the outer sum with alternating sign (-1)^|L|
            # Note: The formula is 1 + Sum ...
            if r % 2 == 1: # |L| is odd -> Subtract (because (-1)^odd = -1)
                total_prob -= term_val
            else:          # |L| is even -> Add
                total_prob += term_val
                
    return total_prob

# --- Test Case ---
p = 5000
s_values = [8, 9, 10, 11, 12, 13, 14, 15]

print(f"Upper Bound for P(B) with p={p}:")
print(f"{'s':<5} | {'P(B) Bound':<20}")
print("-" * 25)

for s in s_values:
    prob = P_super(p, s)
    # Clamp to [0, 1] for sanity, though formula naturally stays within bounds usually
    prob = max(0.0, min(1.0, prob)) 
    print(f"{s:<5} | {prob:.10f}")

Upper Bound for P(B) with p=5000:
s     | P(B) Bound          
-------------------------
8     | 0.9999999778
9     | 0.9996021500
10    | 0.9482858619
11    | 0.5291011849
12    | 0.0869876789
13    | 0.0042362160
14    | 0.0000919863
15    | 0.0000012934


In [19]:
def compare(p, s):
    pb = P(p)
    ps = P_super(p, s)

    print(f"p = {p}, s = {s}")
    print(f"P_base  = {pb:.6e}")
    print(f"P_super = {ps:.6e}")
    print(f"ratio (super/base) = {ps/pb:.6e}")
    print()

for p in [10, 50, 100, 200, 400, 1000, 2000]:
    compare(p, s=15)

p = 10, s = 15
P_base  = 4.710940e-12
P_super = -5.218048e-15
ratio (super/base) = -1.107645e-03

p = 50, s = 15
P_base  = 9.959155e-07
P_super = 7.216450e-15
ratio (super/base) = 7.246046e-09

p = 100, s = 15
P_base  = 1.210907e-04
P_super = -2.442491e-15
ratio (super/base) = -2.017075e-11

p = 200, s = 15
P_base  = 7.543007e-03
P_super = 2.886580e-15
ratio (super/base) = 3.826829e-13

p = 400, s = 15
P_base  = 1.532990e-01
P_super = 1.991740e-13
ratio (super/base) = 1.299252e-12

p = 1000, s = 15
P_base  = 8.510234e-01
P_super = 4.445810e-11
ratio (super/base) = 5.224075e-11

p = 2000, s = 15
P_base  = 9.968164e-01
P_super = 3.612078e-09
ratio (super/base) = 3.623614e-09

